In [8]:
import os

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='deepseek-ai/DeepSeek-V3.2',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

# pydantic
1. 处理数据
2. 验证数据
3. 定义数据格式
4. 虚拟化和反虚拟化
5. 类型转换

In [2]:
from pydantic import BaseModel, Field
from typing import Optional


# 定义一个数据模型
class Person(BaseModel):
    name: Optional[str] = Field(None, description='表示名字')
    hair_color: Optional[str] = Field(None, description='如果知道的画，这个人的头发颜色')
    height_in_meters: Optional[float] = Field(None, description='这个人的身高，单位是米')


In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "你是一个专业的提取算法。只从未结构化文本中提取相关信息。如果你不知道要提取的属性的值，返回该属性的值为null。"),
    ("human", "{input}")
])

In [4]:
from langchain_core.runnables import RunnablePassthrough

chain = {'input': RunnablePassthrough()} | prompt | model.with_structured_output(schema=Person)

In [9]:
text = '马路上走来一个女生，长长的黑头发披在肩上，大概1米7左右，'
resp = chain.invoke(text)
resp

Person(name=None, hair_color='black', height_in_meters=1.7)

In [10]:
# 提取多个人
class ManyPerson(BaseModel):
    people: list[Person] = Field(None, description='表示多个人')

In [11]:
many_chain = {'input': RunnablePassthrough()} | prompt | model.with_structured_output(schema=ManyPerson)

In [12]:
text = "马路上走来一个女生，长长的黑头发披在肩上，大概1米7左右。走在她旁边的是她的男朋友，叫：刘海；比她高10厘米。"
resp = many_chain.invoke(text)
resp

ManyPerson(people=[Person(name=None, hair_color='黑色', height_in_meters=1.7), Person(name='刘海', hair_color=None, height_in_meters=1.8)])